<a id="unit-u.c2ab5542acc141e39187270cfbf4c4bc"></a>

<div>
  <div>11-ЗЕРТХАНАЛЫҚ ЖҰМЫС · APACHE DORIS КӨМЕГІМЕН ДЕРЕКТЕР ҚОЙМАСЫН ҚҰРУ</div>
  <div>Семантикалық көрініс пен белгілерін жариялау</div>
</div>

Бақылау тақтасы күндік тапсырыстарды, модель белгілерді оқиды; екеуіне де жолдары мен мағынасы тұрақты интерфейс қажет. 10-жұмыстың ұсыну кестесінен әр күнге бір жолы бар `bi_order_metrics_l2` семантикалық көрінісін жариялайсыз, тапсырыссыз күн де көрінуі үшін есеп терезесінің әр күніне белгілерді проекциялайсыз, жарияламас бұрын жолдарды, null мәндерін, күндер мен сомаларды тексересіз және бұзылған келісімшарт пен ескірген snapshot-ты әдейі сәтсіз етесіз. Қосымша 5 және 6-бөлімдер жергілікті Superset іске қосып, мысалдың төлем келісімшартында тек сол көріністі оқитын Doris тіркелгісімен оқылатын үш диаграммалы бақылау тақтасын жасайды. Шамамен 70 минут жоспарлаңыз; қосымша бөлімдерге тағы шамамен 15 минут және алғашқы жүктеу кетеді.

**11-модуль: BI және AI тұтынушыларына арналған интерфейстер мен шекаралар** · [Сабақ](course.md) · [Тест](quiz11_bi_and_ai.ipynb) · [Курстың README файлы](../../README.md)


<a id="unit-u.9071a97ef6b04a2a91644e1573a6377d"></a>

<a id="before-you-begin"></a>

## Бастамас бұрын

Алдымен сол дерекқорда 10-зертханалық жұмысты аяқтаңыз. Бұл жұмыс оның `daily_order_metrics_l2` ұсыну кестесін, қосымша бөлімдер `bi_paid_kpis_l2` көрінісін оқиды; екеуін де өзгертпейді.

Жұмыс 12-модуль жариялайтын `bi_order_metrics_l2`, жаттығуда `bi_order_metrics_by_region_l2` көріністерін қайта жасайды; 4-қадам `bi_contract_broken_l2` жасап, қайта жояды.

Қосымша 5 және 6-бөлімдер тек қосқанда орындалады, бақылау тақтасы жұмыс істей беруі үшін зертханалық дерекқордан тыс нысандар жасалып, жұмыстан кейін қалады: `doris-warehousing-bi-superset-1` контейнері мен метадеректер томы бар `doris-warehousing-bi` Compose жобасы, өз компьютеріңізде жиналған `doris-course-superset:4.1.2-pydoris1.1.0` бейнесі және тек `bi_paid_kpis_l2` оқитын `course_bi_<suffix>` Doris тіркелгісі. Оларды жоюды 11-модуль сабағының 11.6 бөлімі айтады. Қосымша бөлімдерді өткізу тестке не 3-деңгейге ештеңе өзгертпейді.

Бұл жұмыс MCP Server орнатпайды, модель шақырмайды; сабақ AI құралының Doris қатынауы қалай шектелетінін түсіндіреді.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import expect, expected_failure
from dw_course.ui import card, show_sql

lab = connect_sandbox(module=11)

<a id="unit-u.103d12d16ee54853b4524a45fb8ab899"></a>

<a id="1-publish-a-semantic-view"></a>

## 1. Семантикалық көріністі жариялау

Семантикалық көрініс — тұтынушылар интерфейсі: тұрақты атаулар, бір жарияланған түйіршіктілік, мағынасы жазылған көрсеткіштер. `bi_order_metrics_l2` 10-жұмыс жолдарын әр күнге бір жолға ірілендіреді: `order_date` — өлшем; `order_count`, `gross_amount` — күндер бойынша қосылатын көрсеткіштер; `average_order_amount` — күн қосындыларынан есептелетін, тапсырыссыз күнде `NULL` болатын қатынас. Бақылау тақтасы қатынасты күндер бойынша қоспауы да, орташаламауы да керек: қосындыларды қосып, қайта бөледі.


In [ ]:
lab.execute("DROP VIEW IF EXISTS bi_order_metrics_l2")
view_ddl = """CREATE VIEW bi_order_metrics_l2 AS
SELECT order_date,
       SUM(order_count) AS order_count,
       SUM(gross_amount) AS gross_amount,
       CASE WHEN SUM(order_count) = 0 THEN NULL
            ELSE SUM(gross_amount) / SUM(order_count) END AS average_order_amount
FROM daily_order_metrics_l2
GROUP BY order_date"""
show_sql("The semantic view", view_ddl)
lab.execute(view_ddl)
lab.sql("SELECT * FROM bi_order_metrics_l2 ORDER BY order_date", title="The semantic view: one row per day")
expect(lab.query("SELECT CAST(order_date AS STRING), order_count, gross_amount, average_order_amount FROM bi_order_metrics_l2 ORDER BY order_date"),
       [("2026-01-01", 9, "1200.00", "133.33"), ("2026-01-02", 1, "110.00", "110.00")],
       title="Two days, as the contract says")

<a id="unit-u.4f69970b89df4f42b30b0a6f5e5e7f65"></a>

**Күтілетін нәтиже**

Анықтама, содан кейін екі жол: 2026-01-01 — 9 тапсырыс, 1200.00, орташа 133.333333; 2026-01-02 — 1 тапсырыс, 110.00, 110.000000. Орташаларды екі ондық таңбаға дейін салыстыратын жасыл белгі.

Көрініс ештеңе сақтамайды: әр сұрау ұсыну кестесін оқиды. Баған атаулары мен мағынасы — келісімшарт; 10-жұмыс кестесі, мысалы тағы баған алып, көрініс тұтынушылары оқитынды өзгертпей өзгере алады.


<a id="unit-u.a45dc8e9a70a4c41ae679a2dbf205860"></a>

<a id="2-project-features-over-the-reporting-window"></a>

## 2. Есеп терезесіне белгілерді проекциялау

Күндік белгілерді оқитын модельге тапсырыссыз күнді қоса әр күнге жол қажет: жоқ жол мен нөл тапсырысты күн бірдей емес, тек екіншісі белгі болады. Семантикалық көріністе тек тапсырыс болған күндер бар. Сондықтан проекция `numbers()` кестелік функциясымен жасалған 2026-01-01–2026-01-03 есеп терезесінің күнтізбесінен басталып, көріністі соған біріктіреді.

Тапсырыссыз күн үшін `zero_order_flag` 1. Ол күнде орташа `NULL` қалады, себебі тапсырыссыз қатынастың мәні жоқ; сандар 0 болады. Проекцияда қанша жол барын, қай күн белгі алатынын болжаңыз.


In [ ]:
features_sql = """WITH calendar AS (
    SELECT DATE_ADD(DATE '2026-01-01', number) AS feature_date FROM numbers("number"="3")
)
SELECT CAST(c.feature_date AS STRING) AS feature_date,
       COALESCE(m.order_count, 0) AS feature_order_count,
       COALESCE(m.gross_amount, 0.00) AS feature_gross_amount,
       m.average_order_amount AS feature_average_amount,
       CASE WHEN COALESCE(m.order_count, 0) = 0 THEN 1 ELSE 0 END AS zero_order_flag
FROM calendar c
LEFT JOIN bi_order_metrics_l2 m ON m.order_date = c.feature_date
ORDER BY c.feature_date"""
lab.sql(features_sql, title="Features for every day of the reporting window")
expect([(row[0], row[1], row[4]) for row in lab.query(features_sql)],
       [("2026-01-01", 9, 0), ("2026-01-02", 1, 0), ("2026-01-03", 0, 1)],
       title="Three days; the day without orders has the flag")

<a id="unit-u.531f9ea6d8334765b6fc313fc73e88c6"></a>

**Күтілетін нәтиже**

Үш жол және жасыл белгі: 2026-01-01 — 9 тапсырыс, 1200.00, орташа 133.333333, белгі 0; 2026-01-02 — 1 тапсырыс, 110.00, 110.000000, белгі 0; 2026-01-03 — 0 тапсырыс, 0.00, орташа жоқ (`NULL`), белгі 1.

Проекция кірістер дайындайды; модель дәлдігі туралы ештеңе айтпайды. 3 қаңтарда тапсырыс болмады ма, әлде әлі келмеді ме, деректер айта алмайды: 4-қадамның өзектілік тексеруі мен конвейердің өз жазбалары айтуы керек.


<a id="unit-u.849728c5bfd547d99e086d9aad553cac"></a>

<a id="3-check-the-interface-before-publication"></a>

## 3. Жарияламас бұрын интерфейсті тексеру

Бақылау тақтасы не модель көріністі оқымас бұрын тұтынушылар сүйенетінді тексеріңіз: күндер саны, ешқашан `NULL` емес көрсеткіштер, алғашқы және соңғы күн, ұсыну кестесімен салыстырылған жалпы тапсырыстар мен сома. Бұл тексерулер интерфейстің келісімшартқа сай болуын талап етеді; диаграмма сүзгілері мен модель мінез-құлқын өз жауаптылары тексереді.


In [ ]:
checks_sql = """SELECT COUNT(*) AS serving_days,
       SUM(CASE WHEN order_count IS NULL OR gross_amount IS NULL THEN 1 ELSE 0 END) AS days_with_nulls,
       CAST(MIN(order_date) AS STRING) AS first_date,
       CAST(MAX(order_date) AS STRING) AS last_date,
       SUM(order_count) AS served_orders,
       SUM(gross_amount) AS served_amount
FROM bi_order_metrics_l2"""
lab.sql(checks_sql, title="The interface's checks")
expect(lab.query(checks_sql), [(2, 0, "2026-01-01", "2026-01-02", 10, "1310.00")], title="The interface passes its checks")
expect(lab.query("SELECT SUM(order_count), SUM(gross_amount) FROM daily_order_metrics_l2"), [(10, "1310.00")],
       title="The view adds up to the service table")

<a id="unit-u.f5f26a3bf93c46dba75d90e391cd4696"></a>

**Күтілетін нәтиже**

Бір жол: 2 ұсынылған күн, null бар күндер 0, алғашқы күн 2026-01-01, соңғы күн 2026-01-02, 10 тапсырыс, 1310.00; екі жасыл белгі, екіншісі 10-жұмыстың ұсыну кестесімен салыстырады.


<a id="unit-u.5581e613e5564a648edfa9a748c27816"></a>

<a id="4-make-a-broken-contract-and-a-stale-snapshot-fail"></a>

## 4. Бұзылған келісімшарт пен ескірген snapshot-ты сәтсіз ету

Орындалатын сұрау келісімшарт сынағы емес. Ұяшық алдымен курсормен көрініс бағандарының атауларын оқып, келісімшарт жариялайтын төртеумен салыстырады. Содан кейін `average_order_amount` жоқ бұзылған көшірме жасайды: сол тексеру оны қабылдамауы керек. Соңында курс тапсырыстары 2026 қаңтарынан болғандықтан, өзектілікті бекітілген оқу күндерімен өлшейді: соңғы күн — 2 қаңтар, 3 қаңтарда бір күн ескі, біркүндік ереже рұқсат етеді, 9 қаңтарда жеті күн ескі, рұқсат етпейді. Нақты тексеру бизнес уақытын және жарияланған ережесін қолданады.


In [ ]:
required_columns = ["order_date", "order_count", "gross_amount", "average_order_amount"]

def published_columns(view_name):
    with lab.connection.cursor() as cursor:
        cursor.execute(f"SELECT * FROM {view_name} LIMIT 0")
        return [column[0] for column in cursor.description]

expect(published_columns("bi_order_metrics_l2"), required_columns, title="The view publishes the four columns of the contract")
lab.execute("DROP VIEW IF EXISTS bi_contract_broken_l2")
try:
    lab.execute("CREATE VIEW bi_contract_broken_l2 AS SELECT order_date, order_count, gross_amount FROM bi_order_metrics_l2")
    with expected_failure("The contract check catches a missing measure", "A consumer that reads average_order_amount would fail on this view"):
        expect(published_columns("bi_contract_broken_l2"), required_columns)
finally:
    lab.execute("DROP VIEW IF EXISTS bi_contract_broken_l2")

freshness_sql = """
SELECT CAST(MAX(order_date) AS STRING) AS latest_date,
       DATEDIFF(%s, MAX(order_date)) AS lag_days
FROM bi_order_metrics_l2
"""
lab.sql(freshness_sql, ("2026-01-03",), title="The lag on January 3")
expect(lab.query(freshness_sql, ("2026-01-03",)), [("2026-01-02", 1)], title="One day old: within the one-day rule")
with expected_failure("The freshness check catches a stale snapshot", "Seven days old on January 9: more than the one-day rule allows"):
    expect(lab.query(freshness_sql, ("2026-01-09",))[0][1] <= 1, True)

<a id="unit-u.d6e046a139b24e278fdd58fa3c1c1761"></a>

**Күтілетін нәтиже**

Төрт карточка. Жасыл белгі: көрініс `order_date`, `order_count`, `gross_amount`, `average_order_amount` жариялайды. Жасыл карточка: келісімшарт тексеруі бұзылған көшірменің жетіспейтін көрсеткішін тапты, көшірме кейін жойылады. 3 қаңтардағы кідіріс: соңғы күн 2026-01-02, 1 күн, жасыл белгі. Жасыл карточка: 9 қаңтарда жеті күн ескі болу ереже рұқсатынан артық.

Күтілетін екі сәтсіздік — осы қадамның мақсаты: ешқашан сәтсіз болмаған тексеру ештеңені тексермеуі мүмкін. Екеуі де `bi_order_metrics_l2` өзгертпеді.


<a id="unit-u.99cefaab91f1417bbb8b47b869bc0254"></a>

<a id="5-optional-connect-a-real-dashboard-to-the-payment-contract"></a>

## 5. Қосымша: нақты бақылау тақтасын төлем келісімшартына қосу

**Бұл және 6-бөлім таңдау бойынша, тек өзіңіз қосқанда орындалады**: келесі ұяшықта `BUILD_DASHBOARD = True` орнатып, орындаңыз. Алғаш бейне жинауға интернет қатынауы, қосымша шамамен 2 GiB Docker жады және хосттың 51888 (`DW_BI_PORT`) порты қажет. Өткізу 11-тестке не 3-деңгейге әсер етпейді; жаттығуға өтіңіз. 11-модуль сабағының 11.6 бөлімі жасалатын нысандар мен жою тәсілін тізеді.

Алдымен `bi_paid_kpis_l2` жасайтын 10-зертханалық жұмыстың 5 және 6-бөлімдерін аяқтаңыз. Ноутбуктегі кесте әлі Business Intelligence (BI) бақылау тақтасы емес. Бұл бөлім **Apache Superset 4.1.2** іске қосып, Doris SQLAlchemy драйверін тіркейді және `bi_paid_kpis_l2` үстінде үш нақты диаграмма жасайды:

1. SQL келісімшартында қайтарымдары бөлек көрсетілген төлем терезесінің GMV-і.
2. Төлем терезесіндегі бірегей төленген тапсырыстар.
3. Тексерілген алымы мен бөлімін қолданатын жасалған тапсырыстар когортының конверсиясы.

Көмекші Superset API арқылы баптайды; әр диаграмма сұрауын Superset арқылы орындап, қайтарылған мәнді де тексереді. Нақты көрсетілімді, метрика SQL-ін және деректер жиынын қарау үшін жасалған бақылау тақтасының URL-ін ашыңыз. API мәндерін тексеруді браузер жайғасуының, жаңарту кестесінің не өндірістік SLA дәлелі деп қабылдамаңыз.

**Қызмет аясы және алғашқы іске қосу шығыны:** курс Superset контейнерінің шегі — 2 GiB RAM және екі CPU, тек `127.0.0.1:51888` мекенжайына байланысады, SQLite метадеректер томын сақтайды. Басқа порт үшін Jupyter-ді `DW_BI_PORT` орнатып іске қосыңыз, мысалы `DW_BI_PORT=61888 .venv/bin/jupyter lab`; көмекші ештеңе іске қоспас бұрын портты тексеріп, бос болмаса, тоқтатылатын бағдарламаны атайды. Алғаш бейне жинауға интернет қатынауы және бірнеше минут қажет болуы мүмкін. Бұл — ашық UI тіркелгі деректері (`course_admin` / `course_local_only`), тұрақты жергілікті құпиясы бар, өндірістік TLS жоқ жергілікті оқу ортасы; оны ашық желіге шығармаңыз. Дерекқор қосылымы `root` не UI әкімшісін емес, бөлек жасалған **тек көріністі оқитын Doris тіркелгісін** қолданады.

Jupyter қашықтағы Linux машинасында болса, Jupyter портымен бірге BI портын да бағыттаңыз: `ssh -N -L 51888:127.0.0.1:51888 user@host`. 51888 бос болмаса, ескі туннельді тоқтатыңыз не басқа жергілікті порт таңдаңыз; қатысы жоқ қызметті тоқтатпаңыз.

Алғаш іске қосу сәтсіз болса, хабарлама себебін айтады: бейне жүктелмеді не жиналмады (Docker машинасына интернет қажет), Docker жады таусылды немесе қызмет 5 минут ішінде жұмысқа жарамды болмады; журнал жиналған егжей-тегжейлі бөлімде.


In [ ]:
# Optional: set to True, then run this cell, to start Superset and build the dashboard.
BUILD_DASHBOARD = False

if BUILD_DASHBOARD:
    from dw_course.bi import build_payment_dashboard
    payment_dashboard = build_payment_dashboard(lab, start=True)
else:
    card("Set BUILD_DASHBOARD = True in this cell and run it again to start Superset and build the dashboard.",
         "skip", "The optional dashboard was not built")

<a id="unit-u.59e02346c6c0402dbc377649a3c8f61a"></a>

**Күтілетін нәтиже**

`BUILD_DASHBOARD = False` болса: сұр карточка, қосымша бақылау тақтасы жасалмады.

`True` болса: төлем келісімшарты орындалғанына жасыл белгі (төленген GMV 660.00, 5 төленген тапсырыс, 8 жасалғанның 5-еуі конверсияланған); қызметті іске қосудың үш қадамы, бәрі жасыл (алғаш іске қосқанда бейне жиналып, бірнеше минут кетеді); бақылау тақтасының Doris тіркелгісі `bi_paid_kpis_l2` оқитынына жасыл белгі (көмекші `dwd_payments_l2` оқи алмайтынын да тексеріп, оқи алса тоқтайды); Superset өзі қайтарған 660.0, 5.0, 0.625 мәндеріне үш жасыл белгі; үш диаграмманың бақылау тақтасына тиесілігіне жасыл белгі; содан кейін бақылау тақтасының `http://127.0.0.1:51888/superset/dashboard/<id>/` URL-і (не `DW_BI_PORT` порты), UI пайдаланушысы `course_admin`, Doris оқырманы `course_bi_<suffix>` және 2026-01-04 00:00:00 шегі.


<a id="unit-u.f660c246a85e47de928e40487864158a"></a>

<a id="6-optional-review-what-the-consumer-can-do"></a>

## 6. Қосымша: тұтынушы не істей алатынын қарау

Жоғарыдағы URL-ді ашып, жергілікті UI тіркелгісімен кіріңіз және үш диаграмманы қараңыз. Конверсия диаграммасы тапсырыс санын емес, пайыз көрсетуі керек. Superset деректер жиыны мен диаграмма редакторларында әр метриканы көрінетін SQL келісімшартына дейін қадағалаңыз. Шек — бекітілген оқу уақыты; бақылау тақтасын жаңарту оны қазіргі уақытқа айналдырмайды.

Жасалған Doris оқырманы тек `bi_paid_kpis_l2` оқиды; көмекші `dwd_payments_l2` тікелей оқуына тыйым салынғанын тексереді. Superset ішінде SQL Lab пен DML өшіру пайдалы, бірақ нақты шекара — дерекқор рұқсаттары. Жергілікті Superset UI әкімшісі конфигурацияны өзгерте алады; бұл көп жалдаушы UI рұқсаттарының көрсетілімі емес.

Қызмет, диаграммалар және тек көріністі оқитын тіркелгі қарауға қолжетімді қалады. Қайта орындау тек осы дерекқордың курс BI нысандарын жаңартып, оқырман құпиясөзін ауыстырады. Курс түбірінен `docker compose --project-name doris-warehousing-bi --file environments/bi/compose.yml stop` арқылы тоқтатыңыз; метадеректер сақталады. Жасалған Doris тіркелгісін тек осы бақылау тақтасын қолданыстан шығарғанда бөлек жойыңыз. Кәдімгі тазалау үшін `down -v` қолданбаңыз.


<a id="unit-u.835568717ae248bfbc39c6764d551773"></a>

<a id="7-practice-add-a-region-filter-without-breaking-consumers"></a>

## 7. Жаттығу: тұтынушыларды бұзбай аймақ сүзгісін қосу

BI тобы күндік тапсырыстарды аймақпен сүзуді сұрайды. `bi_order_metrics_l2` ішінде аймақ жоқ, қосу түйіршіктілігін өзгертеді: жол енді бір күн болмайды. Сол түйіршіктілігі бар ұсыну кестесінен әр күн мен аймаққа бір жолы бар екінші `bi_order_metrics_by_region_l2` көрінісін жариялаңыз. Жолдары бірінші көріністің күндеріне қосылып сәйкес келетінін, бірінші көрініс әлі сол төрт бағанды жариялайтынын тексеріңіз.


In [ ]:
# Your code here

<a id="unit-u.258d7ea0a2174ab2bcb93d6c30d5c9f3"></a>

Келесі ұяшықта үлгі шешім бар; ол жиналған күйде. Салыстыру үшін ашыңыз немесе орындаңыз.


In [ ]:
lab.execute("DROP VIEW IF EXISTS bi_order_metrics_by_region_l2")
lab.execute("""CREATE VIEW bi_order_metrics_by_region_l2 AS
SELECT order_date, region,
       SUM(order_count) AS order_count,
       SUM(gross_amount) AS gross_amount,
       CASE WHEN SUM(order_count) = 0 THEN NULL
            ELSE SUM(gross_amount) / SUM(order_count) END AS average_order_amount
FROM daily_order_metrics_l2
GROUP BY order_date, region""")
lab.sql("SELECT * FROM bi_order_metrics_by_region_l2 ORDER BY order_date, region", title="One row per day and region")
by_day = """SELECT CAST(order_date AS STRING), SUM(order_count), SUM(gross_amount)
FROM bi_order_metrics_by_region_l2 GROUP BY order_date ORDER BY order_date"""
expect(lab.query(by_day), lab.query("SELECT CAST(order_date AS STRING), order_count, gross_amount FROM bi_order_metrics_l2 ORDER BY order_date"),
       title="The regions add up to the days of bi_order_metrics_l2")
expect(published_columns("bi_order_metrics_l2"), required_columns, title="bi_order_metrics_l2 keeps its four columns")

<a id="unit-u.c9cb6140660f4ec6abd09497e0b30d3d"></a>

**Күтілетін нәтиже**

Үш жол: 2026-01-01 EAST — 5 тапсырыс, 720.00, 144.000000; 2026-01-01 WEST — 4, 480.00, 120.000000; 2026-01-02 EAST — 1, 110.00, 110.000000. Екі жасыл белгі: аймақтар бірінші көріністің күндеріне қосылып сәйкес, `bi_order_metrics_l2` төрт бағанын сақтайды.

Жаңа көріністің өз түйіршіктілігі бар, сондықтан екінші келісімшарт: өз жаңартуы, қатынауы және тексерулері қажет, бірінші көріністің бақылау тақталары өзгеріссіз істей береді. Күндік 133.33 орташасы 144.00 пен 120.00 орташасы емес екенін байқаңыз: әр көрініс өз қосындыларын бөледі.


<a id="unit-u.c6ec287691f44fbab5b367a964e28b29"></a>

<a id="you-are-done-when"></a>

## Жұмысты аяқтау шарттары

- `bi_order_metrics_l2` екі күн, 10 тапсырыс, 1310.00 жариялайды, қай бағандары өлшемдер, көрсеткіштер, қатынастар екенін айта аласыз;
- белгі проекциясы неге күнтізбеден басталатынын және 3 қаңтардағы `zero_order_flag` нені айтатынын, нені айтпайтынын түсіндіре аласыз;
- келісімшарт пен өзектілік тексерулерінің әдейі сәтсіз болуын, жарияланған көріністе өтуін көрдіңіз;
- өз түйіршіктілігі бар екінші көрініс тұтынушыларға нені өзгертетінін, нені өзгертпейтінін айта аласыз.


<a id="unit-u.0c3c7092f87f41089a4c210378adc5d0"></a>

<a id="next"></a>

## Келесі қадам

[11-тест: BI және AI тұтынушыларына арналған интерфейстер мен шекаралар](quiz11_bi_and_ai.ipynb) файлын ашыңыз. Бұл — 2-деңгейдің соңғы зертханалық жұмысы; 3-деңгей [12-модуль: жариялау, рұқсаттар және аудит](../../level3/module12-publishing-permissions-audit/course.md) бөлімінен басталады.
